# 레슨 02 — NumPy 심화: 다차원 배열 · 브로드캐스팅 · 시뮬레이션

## 학습 목표

이 레슨을 마치면 다음을 할 수 있다.

1. 2차원 이상의 NumPy 배열을 만들고 `shape`, `ndim`, `size` 를 읽는다.
2. `reshape` · `ravel` · `T` 로 배열의 구조를 자유롭게 바꾼다.
3. 2D 배열에서 행·열 슬라이싱과 팬시 인덱싱을 사용한다.
4. 브로드캐스팅 규칙을 이해하고 연산 가능 여부를 스스로 판단한다.
5. `np.random` 으로 균등분포·정규분포 난수를 생성하고 차이를 설명한다.
6. 표준화(z-점수) 변환을 NumPy 벡터 연산으로 구현한다.
7. 주사위 시뮬레이션으로 대수의 법칙을 수치로 확인한다.
8. 1차원 랜덤워크를 배열 연산으로 구현하고 시각적으로 해석한다.

---

## 1. 왜 다차원 배열인가?

실제 데이터는 거의 항상 **2차원 이상**이다.

| 분야 | 데이터 모양 | 예시 |
|---|---|---|
| 이미지 처리 | (height, width, channels) | (1080, 1920, 3) RGB 사진 |
| 주가 데이터 | (days, stocks) | (252, 5) 5개 종목 1년 |
| 센서 그리드 | (rows, cols) | (20, 30) 건물 층 온도 지도 |
| 머신러닝 특징 | (samples, features) | (10000, 128) 이미지 임베딩 |

레슨 01에서는 1D 배열(벡터)만 다뤘다. 이번 레슨에서는 **2D 배열(행렬)**을 본격적으로 다루며, 브로드캐스팅과 난수를 추가로 배운다.

---

## 2. 2차원 배열 만들기

In [ ]:
import os
import numpy as np

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
DATA_BASE = "https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-data-analysis/lectures/02/data" if IS_COLAB else "./data"
print("data base:", DATA_BASE)

# 직접 생성: 리스트의 리스트를 넘긴다
mat = np.array([[1, 2, 3],
                [4, 5, 6]])   # shape (2, 3) — 2행 3열

print(mat.shape)   # (2, 3)
print(mat.ndim)    # 2   — 축(axis)의 개수
print(mat.size)    # 6   — 전체 원소 수
print(mat.dtype)   # int64

> **용어 정리**
> - `shape` 의 첫 번째 값 → **행(row)** 수 (axis 0)
> - `shape` 의 두 번째 값 → **열(col)** 수 (axis 1)

### 2.1 편의 생성 함수

In [ ]:
zeros = np.zeros((3, 4))        # 0으로 채운 3×4
ones  = np.ones((2, 5))         # 1로 채운 2×5
eye   = np.eye(3)               # 3×3 단위 행렬 (대각선만 1)
full  = np.full((4, 4), 7)      # 7로 채운 4×4

### 2.2 CSV 로드 후 2D 배열 만들기

In [ ]:
# student_scores_2d.csv 에는 class_id, student_id, math, english, science 5열이 있다
raw = np.loadtxt(f"{DATA_BASE}/student_scores_2d.csv", delimiter=",", skiprows=1)
# raw.shape == (120, 5)

class_id   = raw[:, 0].astype(int)
student_id = raw[:, 1].astype(int)
scores     = raw[:, 2:]          # shape (120, 3) — 세 과목 점수만
print(scores.shape)              # (120, 3)

---

## 3. reshape · ravel · T

배열의 **원소 수는 유지**하면서 구조를 바꾼다.

In [ ]:
a = np.arange(12)         # [0, 1, 2, ..., 11]   shape (12,)

b = a.reshape(3, 4)       # shape (3, 4)
c = a.reshape(4, 3)       # shape (4, 3)
d = a.reshape(2, 2, 3)    # shape (2, 2, 3)  — 3D!

# -1 을 쓰면 NumPy 가 나머지 크기를 자동 계산
e = a.reshape(-1, 4)      # shape (3, 4)  — 행 수를 자동으로
f = a.reshape(3, -1)      # shape (3, 4)  — 열 수를 자동으로

In [ ]:
# ravel(): 무조건 1D로 펼친다
g = b.ravel()             # shape (12,)  — b는 (3,4)였음

# .T: 행과 열을 바꾼다 (전치)
h = b.T                   # shape (4, 3)

> **왜 .T 가 필요한가?**  
> 데이터를 분석할 때 "가로로 나열된 특징"을 "세로로 세운 특징"으로 바꿔야 하는 경우가 자주 생긴다. pandas 와 연동할 때도 열/행 방향이 맞아야 올바른 계산이 된다.

---

## 4. 2D 슬라이싱과 인덱싱

2D 배열 슬라이싱은 `[행 범위, 열 범위]` 두 축을 **콤마로 구분**해서 지정한다.

In [ ]:
# scores.shape == (120, 3)  — 0: math  1: english  2: science

# 모든 학생의 수학 점수(열 0)
math_all = scores[:, 0]          # shape (120,)

# 1반 학생(인덱스 0~39)의 수학 점수
math_cls1 = scores[:40, 0]       # shape (40,)

# 처음 5명의 모든 과목
top5 = scores[:5, :]             # shape (5, 3)

# 1반(0~39)의 영어+과학(열 1,2만)
eng_sci_cls1 = scores[:40, 1:]   # shape (40, 2)

### 팬시 인덱싱 (Fancy Indexing)

In [ ]:
# 특정 행을 배열로 지정
rows = [0, 10, 50, 99]
selected = scores[rows, :]       # shape (4, 3)

# 수학 점수가 90점 이상인 학생만 (boolean indexing, 2D 에서도 동일)
mask = scores[:, 0] >= 90
high_math = scores[mask, :]      # 조건 만족 학생 전체 점수
print("90점 이상 학생 수:", high_math.shape[0])

---

## 5. 축(axis) 방향 집계

2D 배열에서 `mean()`, `sum()` 등의 함수는 **axis** 파라미터로 방향을 정한다.

In [ ]:
# scores.shape == (120, 3)
col_mean = scores.mean(axis=0)   # 각 과목 전체 평균  → shape (3,)
row_mean = scores.mean(axis=1)   # 각 학생 세 과목 평균 → shape (120,)
total    = scores.sum(axis=1)    # 각 학생 총점        → shape (120,)

print("과목별 평균:", col_mean.round(2))
print(f"첫 학생 평균: {row_mean[0]:.2f}")

> axis=0 은 "행을 따라" 계산 → 결과가 열 방향.  
> axis=1 은 "열을 따라" 계산 → 결과가 행 방향.  
> 헷갈릴 때는 "axis=0 을 없애는 방향으로 합친다" 고 기억한다.

---

## 6. 브로드캐스팅

브로드캐스팅(broadcasting)은 **크기가 다른 두 배열을 자동으로 맞춰서 연산**하는 NumPy 의 핵심 기능이다.

### 규칙 (3단계)

1. 두 배열의 shape 오른쪽부터 비교한다.
2. 각 차원에서 크기가 같거나, 하나가 1이면 연산 가능하다. **크기가 1인 차원은 상대방 크기로 늘어난다.**
3. 차원 수가 다르면 왼쪽에 1을 자동으로 추가한다.

In [ ]:
a = np.array([[1, 2, 3],
              [4, 5, 6]])     # shape (2, 3)
b = np.array([10, 20, 30])   # shape    (3,)
#                                 → (1, 3) 으로 처리됨

c = a + b
# [[1+10, 2+20, 3+30],
#  [4+10, 5+20, 6+30]]
# [[11, 22, 33],
#  [14, 25, 36]]
print(c.shape)   # (2, 3)

### 실전 예 — 각 학생 점수를 과목 평균으로 빼기

In [ ]:
col_mean = scores.mean(axis=0)      # shape (3,)
centered = scores - col_mean        # shape (120,3) - (3,) → 브로드캐스팅!
# 각 학생의 과목별 편차(deviation)
print(centered[:3].round(2))

### 브로드캐스팅 불가 예

In [ ]:
a = np.ones((3, 4))
b = np.ones((4, 3))
# a + b  → ValueError: shape (3,4) 와 (4,3) 은 맞지 않음
# 이럴 때는 b.T 로 전치한 뒤 연산한다
c = a + b.T   # (3,4) + (3,4) → OK

---

## 7. np.random — 난수 생성

데이터 시뮬레이션, 통계 검정, 머신러닝 가중치 초기화 등 어디서나 쓰이는 핵심 모듈이다.

### 7.1 시드(seed) — 재현 가능한 난수

In [ ]:
np.random.seed(42)    # 이 줄이 있으면 매번 같은 난수 순서
r = np.random.rand(5)
print(r.round(2))
# [0.37 0.95 0.73 0.60 0.16]
# seed(42) 이면 항상 이 값

> **왜 seed 를 고정하는가?**  
> 실험 재현성을 위해서다. 팀원 A 와 B 가 같은 코드를 실행해도 다른 난수를 쓰면 결과가 달라진다. 코드를 공유할 때는 `seed` 를 고정하면 누가 실행해도 동일한 결과를 얻는다.

---

> **🥄 수학 지식 한스푼 — 균등분포 (Uniform Distribution, U(a, b))**
>
> - **뜻**: 구간 [a, b] 사이의 모든 값이 **똑같이** 나올 가능성을 가지는 분포. 어느 값 하나가 더 자주 나오지 않는다.
> - **수식**: 확률밀도 f(x) = 1 / (b − a)  (a ≤ x ≤ b 일 때, 나머지 구간은 0)
> - **읽는 법**: 주사위를 굴렸을 때 1~6 이 모두 같은 확률 → 이산(discrete) 균등분포. `np.random.rand()` 는 [0, 1) 연속 균등분포.
> - **예시**: 응모 추첨, 게임 크리티컬 확률, AB 테스트 사용자 랜덤 배정.

In [ ]:
# [0, 1) 균등분포 — rand
u = np.random.rand(5)             # [0, 1)

# [a, b) 균등분포 — uniform(low, high, size)
u2 = np.random.uniform(10, 20, 5) # [10, 20)

# 정수 균등분포 — randint(low, high, size)  high 는 제외
dice = np.random.randint(1, 7, 10)  # 1~6 주사위 10번

---

> **🥄 수학 지식 한스푼 — 정규분포 (Normal Distribution, N(μ, σ²))**
>
> - **뜻**: 평균(μ) 근처에 몰리고 멀어질수록 급격히 줄어드는 종(bell) 모양 분포. 자연계·사회 현상 대부분이 이 형태를 따른다.
> - **수식**: f(x) = (1 / (σ√2π)) · exp(−(x−μ)² / (2σ²))
> - **읽는 법**: μ=0, σ=1 → **표준정규분포 N(0,1)**. 전체 면적의 68% 가 μ±σ 안에 있고, 95% 가 μ±2σ, 99.7% 가 μ±3σ 안에 있다(68-95-99.7 법칙).
> - **예시**: 키, 몸무게, 시험 점수, 측정 오차, 주식 일간 수익률의 근사.

In [ ]:
# 표준정규분포 N(0,1) — randn
z = np.random.randn(5)

# 임의의 정규분포 N(mu, sigma²) — normal(loc, scale, size)
heights = np.random.normal(loc=170, scale=8, size=1000)  # 평균 170, 표준편차 8

print(f"평균: {heights.mean():.2f}  표준편차: {heights.std():.2f}")
# → 170 ± 8 에 수렴

---

## 8. 표준화 (Standardization)

> **🥄 수학 지식 한스푼 — 표준화와 z-점수 (Standardization & z-score)**
>
> - **뜻**: 서로 단위나 범위가 다른 변수들을 **평균 0, 표준편차 1** 로 맞추는 변환. 국어(100점 만점)와 영어(50점 만점)를 직접 비교할 수 없을 때 표준화하면 같은 기준으로 비교할 수 있다.
> - **수식**: z = (x − μ) / σ
> - **읽는 법**: z = +2 이면 "평균보다 표준편차 2배 높다(상위 약 2.3%)". z = −1 이면 평균보다 한 단계 낮다.
> - **예시**: CSAT(수능) 표준점수, 키 퍼센타일, 머신러닝 특징 전처리(Feature Scaling).

In [ ]:
# 각 과목을 개별적으로 표준화
mu    = scores.mean(axis=0)     # shape (3,) — 과목별 평균
sigma = scores.std(axis=0)      # shape (3,) — 과목별 표준편차

z_scores = (scores - mu) / sigma  # 브로드캐스팅 → shape (120, 3)

print("표준화 후 과목별 평균:", z_scores.mean(axis=0).round(2))
print("표준화 후 과목별 표준편차:", z_scores.std(axis=0).round(2))
# → 모두 [0, 0, 0] 과 [1, 1, 1] 에 근접

표준화는 **머신러닝에서 거의 필수**다. 거리 기반 알고리즘(KNN, SVM, k-means)은 값의 스케일이 크면 그 특징에 지나치게 끌린다. 표준화하면 모든 특징이 공평하게 다뤄진다.

---

## 9. 시뮬레이션 기초 — 주사위와 대수의 법칙

> **🥄 수학 지식 한스푼 — 기댓값과 대수의 법칙 (Expected Value & Law of Large Numbers)**
>
> - **기댓값 뜻**: 어떤 실험을 **무한히 반복**했을 때 결과값의 평균. 기호 E[X] 또는 μ.
> - **수식**: E[X] = Σ xᵢ · P(xᵢ) (이산형) / 주사위 1개 E[X] = (1+2+3+4+5+6)/6 = 3.5
> - **대수의 법칙**: 시행 횟수 n이 커질수록 표본 평균이 기댓값에 **수렴**한다. 100번 주사위를 굴리면 평균이 3.5 가까이 오고, 10,000번 굴리면 더 가까워진다.
> - **예시**: 카지노는 대수의 법칙으로 이익을 보장한다. 수백만 번 게임하면 기댓값대로 수렴하기 때문.

In [ ]:
np.random.seed(0)

# dice_rolls.csv 에서 주사위 데이터를 불러오는 대신 직접 시뮬레이션
for n in [10, 100, 1000, 10000]:
    rolls = np.random.randint(1, 7, n)
    print(f"n={n:>6}  평균 {rolls.mean():.4f}  (기댓값 3.5000)")

예상 출력:
```
n=    10  평균 3.2000  (기댓값 3.5000)
n=   100  평균 3.5200  (기댓값 3.5000)
n=  1000  평균 3.5070  (기댓값 3.5000)
n= 10000  평균 3.4996  (기댓값 3.5000)
```

### 두 주사위 합 분포 분석 (dice_rolls.csv 활용)

In [ ]:
raw_dice = np.loadtxt(f"{DATA_BASE}/dice_rolls.csv", delimiter=",", skiprows=1)
totals = raw_dice[:, 3].astype(int)   # 열 3: total (2~12)

# 각 합의 빈도
for s in range(2, 13):
    cnt = (totals == s).sum()
    pct = cnt / len(totals) * 100
    bar = "█" * round(pct)
    print(f"합 {s:2d}: {cnt:5d}회 ({pct:.1f}%) {bar}")

합 7이 가장 많이 나오는 이유: 7을 만드는 조합이 (1,6),(2,5),(3,4),(4,3),(5,2),(6,1) 으로 6가지로 가장 많기 때문.

---

## 10. 랜덤워크 (Random Walk)

**랜덤워크**는 매 시점마다 +1 또는 −1 등 무작위 변화를 누적하는 모델이다. 물리학에서는 브라운 운동, 금융에서는 주가 모델의 기초로 쓰인다.

In [ ]:
np.random.seed(7)
n_steps = 252          # 영업일 252일
n_paths = 5            # 5개 경로

# step: 매일 +1 or -1 무작위
steps = np.random.choice([-1, 1], size=(n_steps, n_paths))
# cumsum: 누적합 → 위치
walks = np.cumsum(steps, axis=0)     # shape (252, 5)

print("최종 위치:", walks[-1])
print("경로 중 최대 이동:", walks.max(axis=0))
print("경로 중 최소 이동:", walks.min(axis=0))

### 주가 모델 — 기하 브라운 운동(간이)

In [ ]:
raw_stock = np.loadtxt(f"{DATA_BASE}/stock_prices.csv", delimiter=",", skiprows=1)
days   = raw_stock[:, 0].astype(int)
prices = raw_stock[:, 1:]              # shape (252, 5)

# 일간 수익률 (log return)
log_ret = np.diff(np.log(prices), axis=0)   # shape (251, 5)

print("종목별 연환산 수익률(252거래일 기준):")
ann_ret = log_ret.mean(axis=0) * 252
ann_vol = log_ret.std(axis=0)  * np.sqrt(252)
for i, name in enumerate(["TechA","FinB","SmallC","MidD","LargeE"]):
    print(f"  {name}: 수익률 {ann_ret[i]*100:+.1f}%  변동성 {ann_vol[i]*100:.1f}%")

> **일간 로그수익률**: ln(P_t / P_{t-1}). 로그를 쓰는 이유는 음수가 될 수 없는 주가를 다루기 쉽게 만들고, 수익률을 더하는 것이 곱하기와 동치가 되기 때문이다.

---

## 11. 현업 · 대회 활용 사례

### 이미지 처리 파이프라인 (AI/CV 대회)

이미지 한 장은 `shape (H, W, 3)` 의 uint8 배열이다. Kaggle 이미지 분류 대회에서는 아래 전처리가 표준이다.

In [ ]:
rng = np.random.default_rng(123)
img = rng.integers(0, 256, size=(480, 640, 3), dtype=np.uint8)

# img.shape == (480, 640, 3)
img_float = img.astype(np.float32) / 255.0       # [0,1] 정규화
img_norm  = (img_float - 0.5) / 0.5             # [-1,1] 표준화
resized   = img[::2, ::2, :]                     # shape (240, 320, 3) 2배 다운샘플
grayscale = img_float @ np.array([0.299, 0.587, 0.114])  # shape (480, 640)
print(img.shape, resized.shape, grayscale.shape)

### 몬테카를로 시뮬레이션 (리스크 분석)

금융 실무에서 VaR(Value at Risk)를 구할 때 몬테카를로를 사용한다.

In [ ]:
np.random.seed(99)
n_sim = 100_000
mu, sigma = 0.0003, 0.015             # 일간 수익률 파라미터
simulated_returns = np.random.normal(mu, sigma, n_sim)
var_95 = np.percentile(simulated_returns, 5)
print(f"95% VaR: {var_95*100:.2f}%")  # 하루 최대 손실 한도 추정

### 주가 상관관계 분석 (퀀트 전략)

In [ ]:
prices = raw_stock[:, 1:]              # (252, 5)
log_ret = np.diff(np.log(prices), axis=0)
# 상관 행렬: 5×5, 대각선은 1
corr_mat = np.corrcoef(log_ret.T)
print("TechA-FinB 상관계수:", round(corr_mat[0, 1], 3))

---

## 12. 조건 기반 선택 — np.where

`np.where(condition, x, y)` 는 조건이 True 이면 x, False 이면 y 값을 원소마다 선택한다. Python 의 삼항 연산자 `x if cond else y` 를 배열 전체에 적용하는 것과 같다.

In [ ]:
# 점수가 60 이상이면 "pass", 미만이면 "fail"
math_scores = scores[:, 0]
grade_label = np.where(math_scores >= 60, 1, 0)   # 1=pass, 0=fail
print("합격률:", grade_label.mean() * 100, "%")

# 주가가 전날보다 오르면 +1, 내리면 -1 로 변환
changes = np.where(np.diff(prices[:, 0]) > 0, 1, -1)  # shape (251,)
print("상승일 수:", (changes == 1).sum())

`np.clip(array, a_min, a_max)` 는 값을 특정 범위 내로 강제 제한한다.

In [ ]:
# 점수가 0~100 범위를 벗어나는 오류 데이터 보정
cleaned = np.clip(scores, 0, 100)

---

## 13. 자주 하는 실수

| 실수 | 잘못된 코드 | 올바른 코드 |
|---|---|---|
| shape 확인 없이 reshape | `a.reshape(3, 5)` (size=12) | `a.reshape(3, 4)` |
| 브로드캐스팅 방향 혼동 | `(3,4) + (4,)` | `(3,4) + (4,)`는 OK, `(3,4) + (3,)`는 실패 |
| axis 방향 착각 | `scores.mean()` (전체) | `scores.mean(axis=0)` (과목별) |
| seed 미고정 | 실행마다 다른 결과 | `np.random.seed(42)` 첫 줄에 |
| randn vs rand 혼동 | `np.random.rand(n)` for normal | `np.random.randn(n)` or `normal()` |
| cumsum axis 미지정 | `np.cumsum(steps)` (flatten) | `np.cumsum(steps, axis=0)` |

---

## 14. 정리

이번 레슨에서 다룬 핵심 함수/속성:

| 주제 | 핵심 | 의미 |
|---|---|---|
| 2D 배열 | `.shape`, `.ndim`, `.size` | 구조 파악 |
| 변환 | `reshape`, `ravel`, `.T` | 구조 변경 |
| 슬라이싱 | `arr[r1:r2, c1:c2]` | 2D 부분 추출 |
| 축 집계 | `mean(axis=0)`, `sum(axis=1)` | 방향별 집계 |
| 브로드캐스팅 | shape 오른쪽 정렬, 1이면 늘어남 | 자동 크기 맞춤 |
| 난수 | `rand`, `randn`, `normal`, `randint` | 확률 분포별 생성 |
| 표준화 | `(x − μ) / σ` | 스케일 통일 |
| 시뮬레이션 | `cumsum` | 누적 위치 |

## 다음 레슨 예고

레슨 03 에서는 **pandas Series·DataFrame** 을 배운다. NumPy 배열에 **라벨(인덱스/열 이름)**이 붙은 것이 pandas 의 핵심이다. 실무 CSV는 숫자만 있는 게 아니라 날짜, 문자열, 범주형 데이터가 섞여 있는데, 이를 다루는 도구가 pandas다.

---

## 참고 자료

- NumPy 공식 문서 — Broadcasting: <https://numpy.org/doc/stable/user/basics.broadcasting.html>
- NumPy 공식 문서 — Random sampling: <https://numpy.org/doc/stable/reference/random/index.html>
- 3Blue1Brown "Essence of Linear Algebra" (영상): reshape · transpose 개념 시각화

## 데이터 출처

이 레슨의 모든 데이터셋은 교육 목적으로 합성된 가상 데이터이다(CC0). 실제 종목·인물과 무관하다.